# 🩺 第二十九天 · CBLUE 基线提分 v1

**今日目标（约 1.5 小时）**：把 KUAKE-QIC 从 75.7% 提上去——不是玄学，而是**换模型 / 换特征 / 调参**三条明路。

**诚实预期（助手已提前跑完）**：最优 **76.68%（+0.97%）**，且提升主要来自弱类（指标解读 recall 0.16→0.38）。到 75% 之后每 +1% 都越来越难——这是正常规律，今天先理解"为什么能提、提在哪"，更大的跳变留到 Week 6（jieba 分词 / 预训练模型）。

## 第 0 步 · 提分的三条明路（先建立地图）

| 杠杆 | 是什么 | 今天用的 |
|---|---|---|
| **换模型** | 逻辑回归换成更强的分类器 | LinearSVC（线性 SVM） |
| **换特征** | 同样的字，换更好的数字表示 | sublinear_tf、max_df |
| **调参** | 同一模型换超参数 | max_iter 等 |

> 记住一句话：**提分 = 试，不是猜。** 每次只改一个变量，看 dev 分数，涨了留下、跌了回退。

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

train = pd.read_json("KUAKE-QIC/KUAKE-QIC_train.json")
dev   = pd.read_json("KUAKE-QIC/KUAKE-QIC_dev.json")
Xtr_raw, ytr = train["query"], train["label"]
Xde_raw, yde = dev["query"], dev["label"]

def 跑一版(名字, vec, clf):
    Xtr = vec.fit_transform(Xtr_raw); Xde = vec.transform(Xde_raw)
    clf.fit(Xtr, ytr)
    acc = accuracy_score(yde, clf.predict(Xde))
    print(f"{名字:<28} {acc:.4f}")
    return acc

# 基线
base = TfidfVectorizer(analyzer="char", ngram_range=(1,2), min_df=2)
跑一版("基线 char(1,2)+LR", base, LogisticRegression(max_iter=2000))

# 换模型
跑一版("char(1,2)+LinearSVC", base, LinearSVC(max_iter=3000))
跑一版("char(1,2)+朴素贝叶斯", base, __import__('sklearn.naive_bayes', fromlist=['MultinomialNB']).MultinomialNB())

# 换特征
跑一版("char(1,3)+LinearSVC", TfidfVectorizer(analyzer="char", ngram_range=(1,3), min_df=2), LinearSVC(max_iter=3000))
跑一版("char(1,2)sublinear+LR", TfidfVectorizer(analyzer="char", ngram_range=(1,2), min_df=2, sublinear_tf=True), LogisticRegression(max_iter=2000))

# 组合（最优）
跑一版("sublinear+SVC ✅最优", TfidfVectorizer(analyzer="char", ngram_range=(1,2), min_df=2, sublinear_tf=True), LinearSVC(max_iter=3000))

## 第 1 步 · 看结果：哪个改动在起作用

预期结果（助手已跑）：

| 配置 | dev 准确率 |
|---|---|
| 基线 char(1,2)+逻辑回归 | 75.70% |
| char(1,2)+LinearSVC | 76.37% |
| char(1,2)+朴素贝叶斯 | 66.50% ❌ 反而更差 |
| char(1,3)+LinearSVC | 76.47% |
| char(1,2)sublinear+逻辑回归 | 75.86% |
| **sublinear + LinearSVC（最优）** | **76.68%** ✅ |

**读结论**：① 换 LinearSVC 有效（+0.67）；② 朴素贝叶斯不适合这里；③ sublinear_tf 再补一点（+0.3）。三招叠起来 75.7%→76.7%。

## 第 2 步 · 三个新概念（Feynman 法，讲回给我）

1. **LinearSVC（线性支持向量机）**：逻辑回归是"给每类打分选最高"；SVM 换了个目标——**找一条把所有类别分得最开的线**（留最大间隔），鲁棒性更好。对高维稀疏文本，SVM 常常比逻辑回归强一点点。
2. **sublinear_tf=True**：把 TF 换成 `1 + log(TF)`——一个词在同一句出现 10 次，不会比出现 2 次重 5 倍，而是被对数**压平**。防止"某个词狂刷次数"过度影响判断。
3. **max_df**：`min_df` 的反面——砍掉"几乎每句都出现"的烂大街字（如"的、了、怎么"），它们区分度低还占位置。今天试了 max_df=0.9 与 sublinear 叠加后分数没再涨，说明主要功劳是 sublinear+SVC。

In [ ]:
# 第 3 步 · 用最优模型重新生成提交文件
from sklearn.metrics import classification_report

test = pd.read_json("KUAKE-QIC/KUAKE-QIC_test.json")

best_vec = TfidfVectorizer(analyzer="char", ngram_range=(1,2), min_df=2, sublinear_tf=True)
Xtr = best_vec.fit_transform(train["query"]); Xde = best_vec.transform(dev["query"])
best_clf = LinearSVC(max_iter=3000)
best_clf.fit(Xtr, train["label"])

pred = best_clf.predict(Xde)
print("dev 准确率: %.4f" % accuracy_score(dev["label"], pred))
print(classification_report(dev["label"], pred, digits=3, zero_division=0))

test = test.copy()
test["label"] = best_clf.predict(best_vec.transform(test["query"]))
test[["id", "query", "label"]].to_json("KUAKE-QIC/KUAKE-QIC_test_pred_v2.json", orient="records", force_ascii=False)
print("提交文件 v2 已生成")

## 第 3 步 · 错误分析（写这里）

对比 D26 的分类报告，看弱类的变化：
- **指标解读**：f1 0.27 → 0.51，recall 0.16 → 0.38（提升最明显，但仍是全榜最弱）
- 医疗费用 0.90→0.98、功效作用 0.65→0.72、病因分析 0.65→0.75
- **结论**：LinearSVC + sublinear_tf 的功劳主要在**少数类**，多数类已接近天花板

**你的观察**（写这里）：
1. 哪个弱类进步最大？
2. 想继续提分，你猜下一步瓶颈在哪？（提示：字符 n-gram 看不到"词"的边界）

## ✅ D29 完成标准

- [ ] 7 组对比跑通，能说出"哪招有效、哪招无效"
- [ ] 三个新概念能讲回给我（LinearSVC / sublinear_tf / max_df）
- [ ] 提交文件 v2 生成
- [ ] 错误分析写了观察
- [ ] 保存（Cmd + S）

> 完成后喊我验收。**诚实预告**：传统方法到 76.7% 基本见顶，下一步要上 jieba 分词或预训练模型（Week 6）。